# 구조공학 서버 — 두 번째 단계: 한국 설계기준과 재료 물성치 리소스 등록하기

> [!ref] 강의노트 매핑
> - **강의노트 §2.7 단계 ②** — 도메인 자산을 리소스 형태로 노출하는 단계입니다.
> - **선행 학습**: 첫 번째 단계 노트북을 완료하여 누적된 파이썬 모듈 파일에 도구 한 개가 정의된 상태여야 본 노트북을 진행할 수 있습니다.

## 학습 목표
이번 단계에서는 도구만 가지고 있던 서버에 **리소스** 라는 새로운 노출 형태를 추가합니다. 구체적으로 다음 네 가지를 달성합니다.

첫째, 리소스 데코레이터를 사용해 한국 설계기준 요약과 콘크리트 물성치 표와 철근 규격 표 그리고 매개변수가 들어가는 동적 조문 조회 리소스 모두 네 개를 등록합니다. 둘째, 두 가지 표준 자원 식별자 스킴을 정의하고 도메인 자산에 일관된 네이밍 규약을 부여하는 방법을 학습합니다. 셋째, 정적 리소스와 동적 리소스의 차이를 직접 호출 결과로 확인합니다. 정적 리소스는 고정 식별자로 호출되지만, 동적 리소스는 매개변수 패턴을 통해 호출 시점에 다양한 값을 받을 수 있습니다. 넷째, 누적 빌드업을 진행하여 다음 단계에서 검색 보강 생성으로 교체할 준비를 합니다.

## 사전 준비 사항
첫 번째 단계 노트북을 끝까지 실행하여 파이썬 모듈 파일이 생성되어 있어야 본 노트북이 자가완결적으로 동작합니다. 이전 단계에서 등록한 무근 콘크리트 압축강도 검토 도구가 정상 호출되는지 미리 확인해 두면 본 단계 진행 중 발생할 수 있는 환경 문제를 사전에 예방할 수 있습니다.


## §1. 표준 자원 식별자 스킴 설계 — 도메인 자원에 일관된 이름을 붙이는 규약

리소스의 표준 자원 식별자 즉 유알아이는 단순한 식별자가 아니라 **자원의 의미와 계층, 그리고 소유 영역**을 한눈에 드러내는 명함입니다. 본 구조공학 서버에서는 두 가지 스킴을 채택합니다.

첫 번째 스킴은 한국 설계기준을 가리키는 데 사용합니다. 이 스킴은 한국 설계기준의 조문 또는 요약을 가리키는 데 쓰이며, 기준 번호와 조문 번호를 슬래시로 연결한 계층 구조로 표현됩니다. 두 번째 스킴은 재료 물성치 정적 테이블을 가리키는 데 사용합니다. 한국콘크리트학회나 한국 설계기준 표준에 따른 등급별 값을 룩업할 수 있는 테이블입니다.

| 표준 자원 식별자 스킴 | 의미와 적용 범위 | 구체적 예시 |
|---|---|---|
| 한국 설계기준 스킴 | 한국 설계기준의 조문 또는 요약을 가리킵니다. | 콘크리트구조 설계기준 전체 요약, 휨 부재 조문, 전단 부재 조문, 기둥 부재 조문 |
| 재료 데이터 스킴 | 재료 물성치 정적 테이블을 가리킵니다. | 콘크리트 강도 등급별 탄성계수와 인장 균열강도 표, 이형철근 규격별 공칭 직경과 단면적 표 |

> [!tip] 표준 자원 식별자 설계의 세 가지 원칙
> 첫째, 계층 구조를 명시적으로 표현해야 합니다. 슬래시로 자원의 트리 구조를 드러내면 검색이나 필터링이 자연스러워집니다. 예를 들어 기준 번호 다음에 조문 번호가 오는 형태는 마치 책의 목차처럼 자원을 정리해 줍니다. 둘째, 소문자와 하이픈, 슬래시만 사용해야 합니다. 가독성을 높이고, 표준 자원 식별자 규약과 운영 체제를 가리지 않는 호환성을 확보할 수 있습니다. 셋째, 다목적 인터넷 메일 확장 형식 즉 마임 타입을 명시해야 합니다. 자료가 제이슨인지 일반 텍스트인지 명시하면, 클라이언트가 자동으로 파싱 방식을 결정할 수 있어 통합이 매끄러워집니다.

> [!ref] 강의노트 §2.4 (리소스) 인용
> "리소스는 **앱이 제어** 하는 영역이다. 사용자가 명시적으로 컨텍스트 주입 시점을 선택하므로, 도구처럼 거대언어모델이 임의로 호출할 수 없다. 따라서 정적이고 신뢰할 수 있는 도메인 자산을 노출하기에 적합하다. 반대로 매번 결과가 달라지거나 부작용이 있는 작업은 도구로 노출하는 편이 옳다."


## §2. 환경 설정 — 1단계 서버를 다시 구성하기

이번 노트북은 자가완결적으로 동작하도록, S6_st01에서 정의한 도구를 그대로 다시 등록합니다. 실제 운영 환경에서는 `from structural_mcp import mcp` 형태로 직전 단계의 모듈을 불러올 수 있지만, 학습용 노트북에서는 셀 단위로 흐름을 따라가기 쉽도록 다시 정의하는 방식을 택했습니다. 같은 도구를 두 번 등록하는 것이 아니라, 같은 시작점에서 한 단계 더 쌓아 올린다고 이해하시면 됩니다.


In [ ]:
# Stage 2: 도구 + 리소스 누적 빌드업
import json
import math
from mcp.server.fastmcp import FastMCP
from pydantic import Field

mcp = FastMCP("StructuralMCP", log_level="ERROR")


# ── Stage 1 — 도구 (재정의) ──
@mcp.tool()
def check_concrete_strength(
    fck: float = Field(description="콘크리트 설계기준 압축강도 (MPa)"),
    Pu: float = Field(description="소요 축력 (kN)"),
    Ag: float = Field(description="기둥 총 단면적 (mm²)"),
) -> str:
    """무근 콘크리트 기둥의 압축강도를 KDS 41 17 00 기준으로 검토합니다."""
    phi = 0.65
    Pn = 0.85 * fck * Ag / 1000.0
    phi_Pn = phi * Pn
    DCR = Pu / phi_Pn if phi_Pn > 0 else float("inf")
    return json.dumps({
        "fck_MPa": fck,
        "Ag_mm2": Ag,
        "phi_Pn_kN": round(phi_Pn, 2),
        "Pu_kN": Pu,
        "DCR": round(DCR, 3),
        "check": "OK" if DCR <= 1.0 else "NG",
        "reference": "KDS 41 17 00",
    }, indent=2, ensure_ascii=False)


print("✅ Stage 1 도구 재구성 완료")


## §3. 첫 번째 리소스 — `kds://41-17-00/summary` (정적, KDS 기준 요약)

KDS 41 17 00 콘크리트구조 설계기준의 핵심 공식과 강도감소계수를 JSON 형식으로 노출합니다. LLM은 검토 직전에 이 리소스를 읽음으로써 **기준 인용의 정확도** 를 끌어올릴 수 있습니다.

이 리소스에 담긴 핵심 정보는 다음과 같습니다. 휨 부재의 강도감소계수 0.85, 최소 인장철근비 식, 등가 응력블록 깊이 a 산정식이 포함됩니다. 전단 부재의 강도감소계수 0.75와 콘크리트 분담 전단강도 Vc, 전단보강근 분담 Vs 식이 함께 들어갑니다. 무근 압축 부재의 강도감소계수 0.65와 공칭 압축강도 산정식도 포함됩니다.


In [ ]:
# 기존 백업 S6_07.ipynb cell 6 인용 — KDS 요약 리소스
@mcp.resource("kds://41-17-00/summary", mime_type="application/json")
def get_kds_summary() -> str:
    """KDS 41 17 00 (콘크리트구조 설계기준) 주요 내용을 반환합니다."""
    return json.dumps({
        "title": "KDS 41 17 00 콘크리트구조 설계기준",
        "flexure": {
            "phi": 0.85,
            "rho_min": "max(0.25*sqrt(fck)/fy, 1.4/fy)",
            "equivalent_stress_block": "a = As*fy / (0.85*fck*b)",
        },
        "shear": {
            "phi": 0.75,
            "Vc": "(1/6)*sqrt(fck)*b*d",
            "Vs": "Av*fy*d/s",
        },
        "axial_unreinforced": {
            "phi": 0.65,
            "Pn": "0.85*fck*Ag",
        },
    }, indent=2, ensure_ascii=False)


print("✅ kds://41-17-00/summary 등록")


## §4. 두 번째 리소스 — 콘크리트 강도 등급별 물성치 표

한국콘크리트학회와 한국 설계기준에 따른 콘크리트 강도 등급별 물성치를 정리한 리소스입니다. 이십사 등급부터 사십 등급까지 다섯 등급의 설계기준 압축강도와 탄성계수와 인장 균열강도 값을 담고 있어, 거대언어모델이 검토 도구를 호출할 때 정확한 재료 물성치 값을 채워 넣을 수 있도록 룩업 테이블 역할을 합니다.

핵심 산정식을 정리하면 다음과 같습니다. 콘크리트 탄성계수는 팔천오백 곱하기 설계기준 압축강도의 삼분의 일 제곱으로 산정합니다. 단위는 메가파스칼이며, 강도가 높아질수록 탄성계수도 함께 증가하지만 삼분의 일 제곱 관계이므로 비례적으로 늘지는 않는 특성을 가집니다. 인장 균열강도는 영점 육삼 곱하기 설계기준 압축강도의 제곱근으로 산정합니다. 단위는 메가파스칼이며, 콘크리트의 인장저항은 압축강도의 약 십분의 일 수준으로 균열 발생 한계를 결정하는 중요한 값입니다.


In [ ]:
# 기존 백업 S6_07.ipynb cell 6 — 콘크리트 물성치 테이블
@mcp.resource("data://materials/concrete-table", mime_type="application/json")
def get_concrete_table() -> str:
    """콘크리트 강도별 물성치 테이블을 반환합니다."""
    return json.dumps({
        "C24": {"fck": 24, "Ec": 25742, "fr": 3.10},
        "C27": {"fck": 27, "Ec": 26871, "fr": 3.29},
        "C30": {"fck": 30, "Ec": 27924, "fr": 3.46},
        "C35": {"fck": 35, "Ec": 29388, "fr": 3.74},
        "C40": {"fck": 40, "Ec": 30722, "fr": 4.00},
        "note": "Ec = 8500 * fck^(1/3), fr = 0.63*sqrt(fck), 단위 MPa",
    }, indent=2, ensure_ascii=False)


print("✅ data://materials/concrete-table 등록")


## §5. 세 번째 리소스 — 이형철근 규격별 단면적 표

이형철근 디십 규격부터 디삼십이 규격까지 여덟 등급의 공칭 직경과 공칭 단면적 값을 정리한 리소스입니다. 거대언어모델이 휨강도 검토 도구를 호출할 때 인장 철근 단면적 값을 정확히 채워 넣을 수 있도록 룩업 테이블 역할을 합니다.

예를 들어 사용자가 "비일 보에 디이십오 철근 네 개를 배근했다"고 말하면, 거대언어모델은 이 리소스를 조회하여 디이십오 철근 한 본의 단면적이 오백육 점 칠 제곱밀리미터라는 사실을 확인합니다. 그다음 네를 곱하여 총 인장철근 단면적이 이천이십육 점 팔 제곱밀리미터임을 계산할 수 있습니다. 이런 룩업 테이블이 없으면 거대언어모델이 환각을 일으켜 잘못된 단면적 값을 만들어 낼 위험이 큽니다. 룩업 테이블을 리소스로 노출하는 일은 단지 편의 기능이 아니라 모델 출력의 정확성을 보장하는 핵심 안전장치입니다.


In [ ]:
# 기존 백업 S6_07.ipynb cell 6 — 철근 규격 테이블
@mcp.resource("data://materials/rebar-table", mime_type="application/json")
def get_rebar_table() -> str:
    """철근 규격별 단면적 테이블을 반환합니다."""
    return json.dumps({
        "D10": {"db": 9.53, "As": 71.3},
        "D13": {"db": 12.7, "As": 126.7},
        "D16": {"db": 15.9, "As": 198.6},
        "D19": {"db": 19.1, "As": 286.5},
        "D22": {"db": 22.2, "As": 387.1},
        "D25": {"db": 25.4, "As": 506.7},
        "D29": {"db": 28.6, "As": 642.4},
        "D32": {"db": 31.8, "As": 794.2},
        "note": "db: 공칭 직경(mm), As: 공칭 단면적(mm²)",
    }, indent=2, ensure_ascii=False)


print("✅ data://materials/rebar-table 등록")


## §6. 동적 리소스 템플릿 — 매개변수가 들어가는 조문 조회 리소스

표준 자원 식별자 안에 중괄호로 둘러싼 매개변수가 들어 있으면 이를 **동적 리소스 템플릿** 이라고 부릅니다. 호출 시점에 매개변수가 실제 값으로 치환되어, 매번 다른 조문을 반환할 수 있다는 특징이 있습니다.

> [!tip] 정적 리소스와 동적 리소스의 노출 방법 차이
> 정적 리소스는 자원 목록 메서드에서 노출되며, 식별자가 고정되어 있습니다. 예를 들어 한국 설계기준 요약 리소스는 항상 같은 식별자로 호출됩니다. 동적 리소스 즉 리소스 템플릿은 자원 템플릿 목록 메서드에서 노출되며, 매개변수 패턴을 포함한 식별자로 등록됩니다. 클라이언트는 패턴 매칭으로 매개변수 자리에 값을 채워 호출하므로, 같은 등록 한 번으로 무한한 변형을 노출할 수 있습니다.

> [!finding] 이번 단계에서는 임시 구현, 다음 단계에서 검색 보강 생성으로 교체합니다
> 여기서는 학습 단계를 단순화하기 위해 **정적 사전** 에 미리 적어 둔 조문 텍스트를 반환하는 임시 구현을 사용합니다. 다음 노트북에서 이 함수 본체를 다섯 번째 주차에서 만든 검색 보강 생성 체인 호출로 교체하여, 실제 한국 설계기준 조문 코퍼스를 동적으로 검색하게 됩니다. 이 단계 분리는 학습 곡선을 완만하게 만들어 줍니다. 즉 동적 리소스 등록 메커니즘 자체를 먼저 익히고, 그다음 단계에서 백엔드를 검색 시스템으로 교체하는 식입니다.


In [ ]:
# Templated 리소스 — placeholder (S6_st03에서 RAG로 교체)
@mcp.resource("kds://41-17-00/{section_id}", mime_type="text/plain")
def get_kds_section(section_id: str) -> str:
    """KDS 41 17 00의 특정 조문을 반환합니다 (placeholder, S6_st03에서 RAG로 대체)."""
    sections = {
        "4.3": "휨 부재 설계 — 등가 응력블록 a = As*fy/(0.85*fck*b), Mn = As*fy*(d - a/2)",
        "4.4": "전단 부재 설계 — Vc = (1/6)*sqrt(fck)*b*d, Vs = Av*fy*d/s",
        "4.5": "축력 부재 — Pn = 0.85*fck*Ag (무근), φ=0.65",
        "4.6": "최소 철근비 — ρ_min = max(0.25*sqrt(fck)/fy, 1.4/fy)",
    }
    return sections.get(
        section_id,
        f"Section {section_id} not found in KDS 41 17 00 (placeholder)",
    )


print("✅ kds://41-17-00/{section_id} (templated) 등록")


## §7. 검증 단계 — 등록된 리소스 목록 확인하기

지금까지 등록한 정적 리소스 3개와 동적 리소스 템플릿 1개가 서버에 정상적으로 노출되는지 확인합니다. `list_resources()`와 `list_resource_templates()`는 각각 다른 카테고리를 반환하므로 두 메서드를 모두 호출해야 전체 그림이 보입니다. 운영 환경에서 새 리소스를 배포한 직후 이 검증 단계를 거치면, URI 오타나 데코레이터 누락 같은 흔한 실수를 빠르게 잡아낼 수 있습니다.


In [ ]:
import asyncio

async def show_resources():
    static = await mcp.list_resources()
    templates = await mcp.list_resource_templates()

    print(f"📋 정적 리소스 ({len(static)}개):")
    for r in static:
        print(f"  - {r.uri}  [{r.mimeType}]")

    print(f"\n📋 동적 리소스 템플릿 ({len(templates)}개):")
    for t in templates:
        print(f"  - {t.uriTemplate}  [{t.mimeType}]")

await show_resources()


## §8. 리소스 호출 시연 — 정적과 동적을 모두 확인

실제로 리소스를 읽어 봄으로써 등록한 내용이 의도대로 나오는지 확인합니다. 정적 리소스인 한국 설계기준 요약 리소스는 항상 같은 요약 자료를 반환합니다. 반면 동적 리소스인 조문 조회 리소스는 호출 시 넣는 조문 식별자 값에 따라 다른 조문 텍스트를 반환합니다. 이 차이를 직접 눈으로 확인하는 것이 중요합니다.

본 시연에서 살펴볼 두 가지는 다음과 같습니다. 첫째, 정적 리소스 호출에서는 콘크리트구조 설계기준 전체 요약이 반환되며, 휨과 전단과 축력에 대한 강도감소계수와 핵심 산정식이 한 번에 나옵니다. 둘째, 동적 리소스 호출에서는 조문 식별자에 따라 휨 부재 조문 또는 전단 부재 조문이 각각 반환되어, 같은 등록 한 번으로 다양한 결과를 받을 수 있음을 확인할 수 있습니다.


In [ ]:
# Templated 리소스 호출 — section_id 가변
async def test_resources():
    # 정적 리소스
    summary = await mcp.read_resource("kds://41-17-00/summary")
    print("=== KDS 요약 (앞 200자) ===")
    print(str(summary)[:200])

    # 동적 리소스 — 4.3 조문
    sec43 = await mcp.read_resource("kds://41-17-00/4.3")
    print("\n=== KDS 4.3 (휨) ===")
    print(sec43)

    # 동적 리소스 — 4.4 조문
    sec44 = await mcp.read_resource("kds://41-17-00/4.4")
    print("\n=== KDS 4.4 (전단) ===")
    print(sec44)

await test_resources()


## §9. 누적 빌드업 결과를 파일로 갱신 저장하기

첫 단계에서 정의한 도구 한 개와 본 단계에서 새로 등록한 리소스 네 개를 모두 통합한 새 버전을 디스크에 다시 씁니다. 이 파일이 다음 단계 노트북의 입력이 되며, 거기서는 동적 리소스의 임시 구현을 다섯 번째 주차에서 만든 하이브리드 검색 보강 생성 체인으로 교체할 예정입니다. 이렇게 단계마다 파일을 갱신하는 방식은 각 학습 단계의 산출물을 명확히 분리해 주어, 학생이 어떤 단계에서 무엇이 달라졌는지 디스크 변경 사항만 보아도 추적할 수 있게 해 줍니다.

저장되는 파일에는 단계 표시 주석과 다음 단계 예고 주석이 포함되어, 향후 협업자나 미래의 자신이 코드를 다시 읽을 때 문맥을 빠르게 회복할 수 있도록 돕습니다. 본 누적 빌드업 패턴은 학습 단계가 길어질수록 각 단계의 책임 범위를 명확히 분리해 주어 디버깅과 회귀 점검에 큰 도움이 됩니다.


In [ ]:
STAGE2_SOURCE = '''"""structural-mcp — Stage 2 (도구 1 + 리소스 4)

자동 생성: S6_st02_kds_resources.ipynb
다음 단계 S6_st03에서 KDS templated 리소스를 RAG로 교체합니다.
"""
import json
from mcp.server.fastmcp import FastMCP
from pydantic import Field

mcp = FastMCP("StructuralMCP", log_level="ERROR")


# ── Tools ──
@mcp.tool()
def check_concrete_strength(
    fck: float = Field(description="콘크리트 설계기준 압축강도 (MPa)"),
    Pu: float = Field(description="소요 축력 (kN)"),
    Ag: float = Field(description="기둥 총 단면적 (mm²)"),
) -> str:
    """무근 콘크리트 기둥 압축강도 검토 (KDS 41 17 00, φ=0.65)."""
    phi = 0.65
    Pn = 0.85 * fck * Ag / 1000.0
    phi_Pn = phi * Pn
    DCR = Pu / phi_Pn if phi_Pn > 0 else float("inf")
    return json.dumps({
        "phi_Pn_kN": round(phi_Pn, 2), "Pu_kN": Pu,
        "DCR": round(DCR, 3),
        "check": "OK" if DCR <= 1.0 else "NG",
    }, indent=2, ensure_ascii=False)


# ── Resources ──
@mcp.resource("kds://41-17-00/summary", mime_type="application/json")
def get_kds_summary() -> str:
    return json.dumps({
        "title": "KDS 41 17 00",
        "flexure": {"phi": 0.85, "rho_min": "max(0.25*sqrt(fck)/fy, 1.4/fy)"},
        "shear": {"phi": 0.75, "Vc": "(1/6)*sqrt(fck)*b*d"},
    }, indent=2, ensure_ascii=False)


@mcp.resource("data://materials/concrete-table", mime_type="application/json")
def get_concrete_table() -> str:
    return json.dumps({
        "C24": {"fck": 24, "Ec": 25742}, "C27": {"fck": 27, "Ec": 26871},
        "C30": {"fck": 30, "Ec": 27924}, "C35": {"fck": 35, "Ec": 29388},
        "C40": {"fck": 40, "Ec": 30722},
    }, indent=2, ensure_ascii=False)


@mcp.resource("data://materials/rebar-table", mime_type="application/json")
def get_rebar_table() -> str:
    return json.dumps({
        "D10": {"As": 71.3}, "D13": {"As": 126.7}, "D16": {"As": 198.6},
        "D19": {"As": 286.5}, "D22": {"As": 387.1}, "D25": {"As": 506.7},
        "D29": {"As": 642.4}, "D32": {"As": 794.2},
    }, indent=2, ensure_ascii=False)


@mcp.resource("kds://41-17-00/{section_id}", mime_type="text/plain")
def get_kds_section(section_id: str) -> str:
    sections = {
        "4.3": "휨 부재 — a=As*fy/(0.85*fck*b)",
        "4.4": "전단 부재 — Vc=(1/6)*sqrt(fck)*b*d",
    }
    return sections.get(section_id, f"Section {section_id} not found")


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("structural_mcp.py", "w", encoding="utf-8") as f:
    f.write(STAGE2_SOURCE)

print(f"✅ structural_mcp.py Stage 2 저장 ({len(STAGE2_SOURCE)} bytes)")


## §10. 다음 단계 안내 — 세 번째 단계 노트북으로 이어집니다

> [!action] 다음에 학습할 노트북
> 세 번째 단계 노트북에서는 다섯 번째 주차에서 만든 한국 설계기준 검색 보강 생성 체인을 가져와, 본 노트북에서 임시 구현으로 만든 동적 리소스 템플릿을 **실제 검색 결과**로 대체합니다. 동시에 사용자가 임의의 자연어 질의를 넣을 수 있는 자유 검색 리소스도 새로 추가합니다. 이로써 단순 룩업 테이블에서 실제 의미론적 검색으로 한 단계 진화하게 됩니다.

> [!finding] 본 단계와 다음 단계의 변화 미리보기 — 정적에서 동적으로
> 본 단계에서 동적 리소스의 구현은 단순한 사전 룩업입니다. 즉 미리 등록해 둔 조문 식별자만 조회 가능하며, 새로운 질의가 들어오면 사전에 없는 한 빈 결과를 반환합니다. 다음 단계에서는 이를 빈도 기반 텍스트 검색과 벡터 임베딩 기반 의미 검색을 결합한 하이브리드 검색으로 대체합니다. 이렇게 하면 자유로운 자연어 질의가 가능해지며, 사용자가 등록되지 않은 표현으로 질문하더라도 가장 관련 높은 조항을 찾아 반환할 수 있게 됩니다. 도메인 자산을 한 번 만들어 두면 여러 인터페이스로 반복 노출할 수 있다는 누적 효과를 본격적으로 체험하는 단계가 됩니다.
